In [1]:
import pandas as pd
import numpy as np
import warnings
from pathlib import Path
import sys
import os
from datetime import datetime

# Most reliable approach - resolves relative to the notebook file itself
# Walk up from cwd until we find the project root (identified by a known file)
project_root = Path.cwd()
while not (project_root / "src").exists():
    project_root = project_root.parent
project_root_str = str(project_root)

if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

os.chdir(project_root_str)

from src.utils.team_info import *
from src.utils.helper_functions import *
from src.pipeline.props_pipeline.ppm_pipeline import *
from src.pipeline.props_pipeline.apm_pipeline import *
from src.pipeline.props_pipeline.rpm_pipeline import *
from src.pipeline.props_pipeline.min_pipeline import *
from src.live import *
from src.historical_analysis.dataScraper import *

warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)

### Get updated lineups

In [2]:
from src.utils.scrap_starters import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
print("\nQuestionable Players:")
print(scraper.getQuestionablePlayers())
print("\nOut Players:")
print(scraper.getOutPlayers())
outPlayers = scraper.getOutPlayers()
scraper.updateTeamInfo()  # Update teamInfo.py


Questionable Players:
{'CLE': ['Dean Wade', 'Donovan Mitchell'], 'DET': ['Cade Cunningham', 'Isaiah Stewart', 'Kevin Huerter'], 'MEM': ['Jahmai Mashack'], 'SAS': ['Emanuel Miller'], 'PHX': ['Grayson Allen']}

Out Players:
{'ATL': ['Jock Landale'], 'CLE': ['Thomas Bryant'], 'MIL': ['Bobby Portis', 'Myles Turner', 'Kyle Kuzma', 'Giannis Antetokounmpo'], 'MIN': ['Julius Randle', 'Mike Conley', 'Ayo Dosunmu', 'Rudy Gobert', 'Anthony Edwards'], 'ORL': ['Jonathan Isaac', 'Jett Howard'], 'MEM': ['Cam Spencer', 'Javon Small', 'GG Jackson', 'Ty Jerome'], 'DEN': ['Spencer Jones', 'Peyton Watson'], 'POR': ['Jerami Grant'], 'SAS': ['Stephon Castle', 'Victor Wembanyama'], 'LAC': ['Darius Garland', 'Isaiah Jackson'], 'PHX': ['Mark Williams', 'Haywood Highsmith']}
Note: CLE (Cavaliers) has 4 confirmed players - lineup will still be updated
Note: DET (Pistons) has 4 confirmed players - lineup will still be updated
Note: DAL (Mavericks) has 4 confirmed players - lineup will still be updated
Successful

### Dataset

In [3]:
s25 = pd.read_csv('data/raw/season_stats/S25.csv').sort_values(by='GAME_DATE')
s26 = pd.read_csv('data/raw/season_stats/S26.csv').sort_values(by='GAME_DATE')
base_df = pd.concat([s25, s26])
base_df.tail()

,Unnamed: 0,SEASON_YEAR,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,TOV,STL,BLK,BLKA,PF,PFD,PTS,PLUS_MINUS,NBA_FANTASY_PTS,DD2,TD3,WNBA_FANTASY_PTS,AVAILABLE_FLAG,MIN_SEC,TEAM_COUNT,E_OFF_RATING,OFF_RATING,sp_work_OFF_RATING,E_DEF_RATING,DEF_RATING,sp_work_DEF_RATING,E_NET_RATING,NET_RATING,sp_work_NET_RATING,AST_PCT,AST_TO,AST_RATIO,OREB_PCT,DREB_PCT,REB_PCT,TM_TOV_PCT,E_TOV_PCT,EFG_PCT,TS_PCT,USG_PCT,E_USG_PCT,E_PACE,PACE,PACE_PER40,sp_work_PACE,PIE,POSS,FGM_PG,FGA_PG,TEAM_FGM,TEAM_FGA,TEAM_FG_PCT,TEAM_FG3M,TEAM_FG3A,TEAM_FG3_PCT,TEAM_FTM,TEAM_FTA,TEAM_FT_PCT,TEAM_OREB,TEAM_DREB,TEAM_REB,TEAM_AST,TEAM_TOV,TEAM_STL,TEAM_BLK,TEAM_BLKA,TEAM_PF,TEAM_PFD,TEAM_PTS,TEAM_PLUS_MINUS,TEAM_E_OFF_RATING,TEAM_OFF_RATING,TEAM_E_DEF_RATING,TEAM_DEF_RATING,TEAM_E_NET_RATING,TEAM_NET_RATING,TEAM_AST_PCT,TEAM_AST_TO,TEAM_AST_RATIO,TEAM_OREB_PCT,TEAM_DREB_PCT,TEAM_REB_PCT,TEAM_TM_TOV_PCT,TEAM_EFG_PCT,TEAM_TS_PCT,TEAM_E_PACE,TEAM_PACE,TEAM_PACE_PER40,TEAM_POSS,TEAM_PIE,OPP_TEAM_ID,OPP_OPP_ABBREVIATION_base,OPP_OPP_NAME_base,OPP_FGM,OPP_FGA,OPP_FG_PCT,OPP_FG3M,OPP_FG3A,OPP_FG3_PCT,OPP_FTM,OPP_FTA,OPP_FT_PCT,OPP_OREB,OPP_DREB,OPP_REB,OPP_AST,OPP_TOV,OPP_STL,OPP_BLK,OPP_BLKA,OPP_PF,OPP_PFD,OPP_PTS,OPP_PLUS_MINUS,OPP_E_OFF_RATING,OPP_OFF_RATING,OPP_E_DEF_RATING,OPP_DEF_RATING,OPP_E_NET_RATING,OPP_NET_RATING,OPP_AST_PCT,OPP_AST_TO,OPP_AST_RATIO,OPP_OREB_PCT,OPP_DREB_PCT,OPP_REB_PCT,OPP_TM_TOV_PCT,OPP_EFG_PCT,OPP_TS_PCT,OPP_E_PACE,OPP_PACE,OPP_PACE_PER40,OPP_POSS,OPP_PIE,START_POSITION,pos,age
140,NaN,2025-26,1630598,Aaron Wiggins,Aaron,1610612760,OKC,Oklahoma City Thunder,22501155,2026-04-07T00:00:00,OKC @ LAL,W,19.950000,4,7,0.571,2,4,0.500,0,0,0.0,0,2,2,0,1,1,0,0,2,0,10,4,14.4,0,0,16.0,1,19:57,1,109.7,111.6,111.6,103.3,102.3,102.3,6.4,9.3,9.3,0.000,0.00,0.0,0.000,0.091,0.044,12.5,12.5,0.714,0.714,0.163,0.168,103.89,103.46,86.22,103.46,0.077,43,4.0,7.0,45,89,0.506,21,41,0.512,12,14,0.857,10,36,46,29,12.0,15,2,3,21,12,123,36.0,126.6,128.1,91.0,89.7,35.6,38.4,0.644,2.42,21.2,0.273,0.776,0.538,0.125,0.624,0.646,96.4,96.5,80.42,96,0.655,1610612747,LAL,Los Angeles Lakers,32,73,0.438,9,26,0.346,14,31,0.452,8,30,38,26,17.0,7,3,2,12,21,87,-36.0,91.0,89.7,126.6,128.1,-35.6,-38.4,0.813,1.53,19.4,0.224,0.727,0.462,0.175,0.500,0.502,96.4,96.5,80.42,97,0.345,NaN,SG,27.0
139,NaN,2025-26,1642347,Jamal Shead,Jamal,1610612761,TOR,Toronto Raptors,22501151,2026-04-07T00:00:00,TOR vs. MIA,W,23.560000,1,4,0.250,1,3,0.333,0,0,0.0,0,1,1,11,3,0,0,0,2,1,3,27,17.7,0,0,16.0,1,23:34,1,129.4,128.0,128.0,71.6,71.2,71.2,57.8,56.8,56.8,0.423,3.67,61.1,0.000,0.031,0.018,16.7,16.7,0.375,0.375,0.119,0.122,102.97,103.90,86.59,103.90,0.071,50,1.0,4.0,49,99,0.495,12,28,0.429,11,16,0.688,15,41,56,34,12.0,9,5,3,19,15,121,26.0,117.4,118.6,91.0,93.1,26.4,25.5,0.694,2.83,22.2,0.315,0.759,0.545,0.118,0.556,0.571,103.7,102.0,85.00,102,0.634,1610612748,MIA,Miami Heat,33,91,0.363,12,44,0.273,17,19,0.895,10,34,44,26,15.0,3,3,5,15,19,95,-26.0,91.0,93.1,117.4,118.6,-26.4,-25.5,0.788,1.73,18.3,0.241,0.685,0.455,0.147,0.429,0.478,103.7,102.0,85.00,102,0.366,NaN,PG,23.0
138,NaN,2025-26,203914,Gary Harris,Gary,1610612749,MIL,Milwaukee Bucks,22501150,2026-04-07T00:00:00,MIL @ BKN,L,21.101667,3,9,0.333,1,4,0.250,0,0,0.0,1,2,3,1,0,0,2,1,1,1,7,-11,18.1,0,0,16.0,1,21:06,1,80.0,77.8,77.8,105.7,104.5,104.5,-25.7,-26.8,-26.8,0.100,0.00,10.0,0.043,0.105,0.071,0.0,0.0,0.389,0.389,0.170,0.185,99.27,101.22,84.35,101.22,0.069,45,3.0,9.0,35,81,0.432,15,45,0.333,5,8,0.625,9,35,44,25,20.0,6,7,4,20,13,90,-6.0,94.2,96.8,99.9,103.2,-5.7,-6.5,0.714,1.25,18.8,0.313,0.814,0.549,0.215,0.525,0.532,95.8,93.0,77.50,93,0.464,1610612751,BKN,Brooklyn Nets,33,75,0.440,9,26,0.346,21,23,0.913,3,29,32,18,14.0,11,4,7,13,20,96,6.0,99.9,103.2,94.2,96.8,5.7,6.5,0.545,1.29,15.1,0.186,0.688,0.451,0.151,0.500,0.564,95.8,93.0,77.50,93,0.536,NaN,SG,31.0
149,NaN,2025-26,1642967,John Poulakidas,John,1610612742,DAL,Dallas Mav

### Load latest odds on file

In [4]:
def get_latest_file(pattern):
    files = list(Path('data/raw/team_lines').glob(pattern))
    return max(files, key=lambda f: f.stat().st_mtime) if files else None

file = get_latest_file('NBA_*.json')
if file is None:
    raise ValueError("No JSON file found")

# try normal load first
try:
    team_dds = pd.read_json(file)
except ValueError:
    # fallback for nested JSON
    import json
    with open(file) as f:
        data = json.load(f)
    team_dds = pd.json_normalize(data)

print("Loaded:", file.name)
team_dds.head()

Loaded: NBA_20260408_145444.json


,home_team,away_team,commence_time,bookmakers
0,Cleveland Cavaliers,Atlanta Hawks,2026-04-08 23:10:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
1,Detroit Pistons,Milwaukee Bucks,2026-04-08 23:10:00+00:00,"[{'bookmaker': 'DraftKings', 'last_updated': '..."
2,Orlando Magic,Minnesota Timberwolves,2026-04-08 23:10:00+00:00,"[{'bookmaker': 'DraftKings', 'last_updated': '..."
3,San Antonio Spurs,Portland Trail Blazers,2026-04-09 00:10:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
4,Denver Nuggets,Memphis Grizzlies,2026-04-09 01:10:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."


In [7]:
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

def get_latest_file(pattern):
    files = list(Path('data/raw/player_lines').glob(pattern))
    if not files:
        return None
    return max(files, key=lambda p: p.stat().st_mtime)

us_file = get_latest_file(f'NBA_US_{today}*.csv')
dfs_file = get_latest_file(f'NBA_DFS_{today}*.csv')

#load season stats
pts_df = pd.read_csv('data/processed/training/S26_TRAINING_PPM.csv')
ast_df = pd.read_csv('data/processed/training/S26_TRAINING_APM.csv')
reb_df = pd.read_csv('data/processed/training/S26_TRAINING_RPM.csv')
min_df = pd.read_csv('data/processed/training/S26_TRAINING_MIN.csv')

#load dfs lines
lines_dfs = pd.read_csv(dfs_file)
lines_dfs_pts = lines_dfs[(lines_dfs['CATEGORY'] == 'player_points')]
lines_dfs_ast = lines_dfs[(lines_dfs['CATEGORY'] == 'player_assists')]
lines_dfs_reb = lines_dfs[(lines_dfs['CATEGORY'] == 'player_rebounds')]
pts_names = lines_dfs_pts['NAME'].unique()
ast_names = lines_dfs_ast['NAME'].unique()
reb_names = lines_dfs_reb['NAME'].unique()

#load us lines with actual odds
lines_us = pd.read_csv(us_file)
lines_us_pts = lines_us[(lines_us['CATEGORY'] == 'player_points')]
lines_us_ast = lines_us[(lines_us['CATEGORY'] == 'player_assists')]
lines_us_reb = lines_us[(lines_us['CATEGORY'] == 'player_rebounds')]

print(f"DFS latest pull: {lines_dfs['DATA_PULLED_AT'].max()}")
print(f"US latest pull: {lines_us['DATA_PULLED_AT'].max()}")
lines_dfs_pts.head()

DFS latest pull: 2026-04-08 14:54:44
US latest pull: 2026-04-08 14:54:07


,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE,DATA_PULLED_AT
0,PrizePicks,player_points,Donovan Mitchell,Over,24.5,-137,2026-04-08,2026-04-08T21:54:07Z,2026-04-08 14:54:44
1,PrizePicks,player_points,Donovan Mitchell,Under,24.5,-137,2026-04-08,2026-04-08T21:54:07Z,2026-04-08 14:54:44
2,PrizePicks,player_points,Jalen Johnson,Over,22.5,-137,2026-04-08,2026-04-08T21:54:07Z,2026-04-08 14:54:44
3,PrizePicks,player_points,Jalen Johnson,Under,22.5,-137,2026-04-08,2026-04-08T21:54:07Z,2026-04-08 14:54:44
4,PrizePicks,player_points,Nickeil Alexander-Walker,Over,20.5,-137,2026-04-08,2026-04-08T21:54:07Z,2026-04-08 14:54:44


### Load my models

In [8]:
import joblib

#minutes
min_bundle = joblib.load("src/models/saved_models/min_quantile_xgb.joblib")
min_quantile_models = min_bundle["quantile_models"]
min_feature_names = min_bundle["feature_names"]

#points per minute
ppm_bundle = joblib.load("src/models/saved_models/ppm_quantile_xgb.joblib")
ppm_quantile_models = ppm_bundle["quantile_models"]
ppm_feature_names = ppm_bundle["feature_names"]

#assists per minute
apm_bundle = joblib.load("src/models/saved_models/apm_quantile_xgb.joblib")
apm_quantile_models = apm_bundle["quantile_models"]
apm_feature_names = apm_bundle["feature_names"]

#rebounds per minute
rpm_bundle = joblib.load("src/models/saved_models/rpm_quantile_xgb.joblib")
rpm_quantile_models = rpm_bundle["quantile_models"]
rpm_feature_names = rpm_bundle["feature_names"]

### Get Min predictions and Stat Per Min predictions 

In [9]:
pts_preds = predict_min_times_rate(
    pts_names, min_df, pts_df, current_date,
    name_dict=nameDict,
    rate_pipeline=ppm_pipeline,
    rate_quantile_models=ppm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="PTS",
)
ast_preds = predict_min_times_rate(
    ast_names, min_df, ast_df, current_date,
    name_dict=nameDict,
    rate_pipeline=apm_pipeline,
    rate_quantile_models=apm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="AST",
)
reb_preds = predict_min_times_rate(
    reb_names, min_df, reb_df, current_date,
    name_dict=nameDict,
    rate_pipeline=rpm_pipeline,
    rate_quantile_models=rpm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="REB",
)
ast_preds.head(10)

[SKIP] A.J. Green: single positional indexer is out-of-bounds
[SKIP] Wendell Carter Jr: single positional indexer is out-of-bounds
[SKIP] Derrick Jones: single positional indexer is out-of-bounds
[SKIP] Wendell Carter Jr: single positional indexer is out-of-bounds
[SKIP] Derrick Jones: single positional indexer is out-of-bounds


,PLAYER_NAME,MARKET,MIN_Q10,MIN_Q50,MIN_Q90,RATE_Q10,RATE_Q50,RATE_Q90,STAT_Q10,STAT_Q50,STAT_Q90,RATE_HISTORY
0,James Harden,AST,28.42,34.85,42.21,0.1772,0.2374,0.3656,5.04,8.27,15.43,"[0.1978518937252685, 0.2608922515001304, 0.171..."
1,Dyson Daniels,AST,22.65,30.39,36.94,0.0765,0.1436,0.2482,1.73,4.36,9.17,"[0.2678913126674321, 0.0888099467140319, 0.115..."
2,Donovan Mitchell,AST,26.66,31.00,38.32,0.0523,0.1583,0.2430,1.39,4.91,9.31,"[0.1422475106685633, 0.1275917065390749, 0.149..."
3,CJ McCollum,AST,24.36,30.82,38.45,0.0533,0.1556,0.2547,1.30,4.79,9.80,"[0.2161828289067325, 0.1428571428571428, 0.066..."
4,Cade Cunningham,AST,20.07,29.72,37.49,0.1311,0.2848,0.4119,2.63,8.46,15.44,"[0.298804780876494, 0.3379256563556018, 0.4301..."
5,Jalen Suggs,AST,20.27,27.95,35.49,0.0725,0.1681,0.2718,1.47,4.70,9.65,"[0.2223634053367217, 0.1366120218579234, 0.084..."
6,Paolo Banchero,AST,28.61,34.54,40.59,0.0605,0.1456,0.2459,1.73,5.03,9.98,"[0.1032346868547832, 0.1891891891891892, 0.118..."
7,Donte DiVincenzo,AST,21.32,27.37,35.10,0.0637,0.0918,0.2262,1.36,2.51,7.94,"[0.0752445447705041, 0.1401050788091068, 0.063..."
8,Tristan da Silva,AST,16.26,24.76,31.62,0.0897,0.0601,0.1423,1.46,1.49,4.50,"[0.1836884643644379, 0.087514585764294, 0.0486..."
9,De'Aaron Fox,AST,22.09,30.60,38.75,0.1062,0.1713,0.3118,2.35,5.24,12.08,"[0.0925354719309068, 0.2973240832507433, 0.312..."


### Get Line Probabilities

In [10]:
all_line_probs = pd.concat([
    line_probs_for_market(ast_preds, lines_dfs_ast, nameDict, run_stat_simulation),
    line_probs_for_market(reb_preds, lines_dfs_reb, nameDict, run_stat_simulation),
    line_probs_for_market(pts_preds, lines_dfs_pts, nameDict, run_pts_simulation),
], ignore_index=True)
all_line_probs.sample(10)

,PLAYER_NAME,MARKET,LINE,MIN_Q50,STAT_Q50,P_OVER,P_UNDER
46,Paolo Banchero,REB,8.5,34.54,7.87,0.423,0.577
168,Julius Randle,PTS,21.5,32.90,22.14,0.462,0.538
79,Sam Merrill,REB,2.5,22.71,2.70,0.492,0.508
66,Marvin Bagley III,REB,6.5,19.97,6.31,0.447,0.553
130,Donovan Clingan,PTS,12.5,25.89,10.50,0.298,0.702
86,Bruce Brown,REB,4.5,23.58,3.22,0.314,0.686
77,Jordan Miller,REB,3.5,20.34,2.67,0.321,0.679
20,Shai Gilgeous-Alexander,AST,6.0,34.30,6.14,0.439,0.455
123,De'Aaron Fox,PTS,21.5,30.60,18.51,0.357,0.643
51,Donovan Clingan,REB,11.5,25.89,10.97,0.466,0.534


In [11]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_dds, nameDict,
    line_bookmaker='Underdog',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

underdog_all_lines = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left').dropna()
underdog_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q50,STAT_Q50,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
35,Jordan Goodwin,AST,2.5,25.79,2.75,0.414,0.586,AST,Underdog,Dallas Mavericks,-12.0,233.0,115.2,19.0,102.65,4.0,-137.0,-137.0,0.578,0.578,2.0,1.5,2.45,-0.5,-1.0,0.204,0.419,0.581,-27.52,0.51,0.2,0.2,0.20,0.30,25.53,5.79,0.14,0.05,1.25,4.0
17,Christian Braun,AST,2.5,27.14,2.52,0.423,0.577,AST,Underdog,Memphis Grizzlies,-23.5,246.5,118.1,26.0,101.45,10.0,-105.0,-103.0,0.512,0.507,2.3,2.0,1.83,-0.2,-0.5,0.109,0.457,0.543,-10.78,7.02,0.4,0.4,0.47,0.47,32.48,6.26,0.15,0.04,4.40,5.0
145,Bruce Brown,PTS,8.5,23.58,9.56,0.698,0.302,PTS,Underdog,Memphis Grizzlies,-23.5,246.5,118.1,26.0,101.45,10.0,-109.0,-114.0,0.522,0.533,8.5,8.0,4.28,0.0,-0.5,0.000,0.500,0.500,-4.13,-6.14,0.4,0.5,0.47,0.42,21.21,4.98,0.17,0.05,6.60,5.0
25,Jamal Murray,AST,6.5,34.14,6.24,0.471,0.528,AST,Underdog,Memphis Grizzlies,-23.5,246.5,118.1,26.0,101.45,10.0,100.0,-114.0,0.500,0.533,7.8,7.0,3.19,1.3,0.5,-0.408,0.658,0.342,31.60,-35.80,0.8,0.6,0.47,0.46,38.57,4.05,0.24,0.06,7.83,6.0
67,Jordan Goodwin,REB,6.5,25.79,5.47,0.396,0.604,REB,Underdog,Dallas Mavericks,-12.0,233.0,115.2,19.0,102.65,4.0,-130.0,-105.0,0.565,0.512,6.0,7.0,2.45,0.5,1.5,-0.204,0.581,0.419,2.79,-18.20,0.2,0.6,0.47,0.33,25.53,5.79,0.14,0.05,3.25,4.0


In [12]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_dds, nameDict,
    line_bookmaker='PrizePicks',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

prizePicks_all_lines = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left').dropna()
prizePicks_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q50,STAT_Q50,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
159,Chet Holmgren,PTS,14.5,28.51,16.88,0.514,0.486,PTS,PrizePicks,Los Angeles Clippers,-8.0,225.0,114.9,17.0,97.33,28.0,-137.0,-137.0,0.578,0.578,14.8,15.5,4.52,-0.2,0.5,0.044,0.482,0.518,-16.62,-10.39,0.4,0.5,0.53,0.51,26.20,5.20,0.21,0.03,14.33,3.0
144,Taylor Hendricks,PTS,11.5,25.50,9.84,0.333,0.667,PTS,PrizePicks,Denver Nuggets,23.5,246.5,116.1,21.0,99.42,20.0,-109.0,-125.0,0.522,0.556,9.6,10.0,4.48,-0.9,-0.5,0.201,0.420,0.580,-19.47,4.40,0.2,0.5,0.47,0.30,23.92,5.37,0.18,0.06,9.33,3.0
20,Shai Gilgeous-Alexander,AST,6.0,34.30,6.14,0.439,0.455,AST,PrizePicks,Los Angeles Clippers,-8.0,225.0,114.9,17.0,97.33,28.0,-137.0,-137.0,0.578,0.578,6.0,6.5,1.70,0.0,0.5,0.000,0.500,0.500,-13.50,-13.50,0.6,0.5,0.53,0.48,31.17,5.39,0.32,0.07,8.67,6.0
8,Tristan da Silva,AST,1.5,24.76,1.49,0.484,0.515,AST,PrizePicks,Minnesota Timberwolves,-10.5,231.5,111.9,5.0,101.46,9.0,-137.0,-137.0,0.578,0.578,2.0,1.5,2.11,0.5,0.0,-0.237,0.594,0.406,2.76,-29.76,0.4,0.5,0.47,0.45,27.59,6.16,0.17,0.04,2.00,3.0
129,Keldon Johnson,PTS,14.5,26.70,16.03,0.563,0.437,PTS,PrizePicks,Portland Trail Blazers,-3.5,229.0,113.7,13.0,101.76,7.0,108.0,-125.0,0.481,0.556,14.4,14.0,5.21,-0.1,-0.5,0.019,0.492,0.508,2.34,-8.56,0.2,0.5,0.40,0.35,23.66,2.99,0.23,0.05,9.20,5.0


In [13]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_dds, nameDict,
    line_bookmaker='Betr DFS',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

betr_all_lines = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left').dropna()
betr_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q50,STAT_Q50,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
116,Naz Reid,PTS,17.5,26.27,15.25,0.195,0.805,PTS,Betr DFS,Orlando Magic,10.5,231.5,113.9,14.0,100.41,14.0,-120.0,-108.0,0.545,0.519,11.6,11.5,4.30,-4.9,-5.0,1.140,0.127,0.873,-76.72,68.13,0.4,0.2,0.20,0.33,25.96,3.81,0.22,0.04,12.33,3.0
47,Jalen Suggs,REB,4.5,27.95,3.79,0.431,0.569,REB,Betr DFS,Minnesota Timberwolves,-10.5,231.5,111.9,5.0,101.46,9.0,101.0,-111.0,0.498,0.526,4.5,4.0,2.22,0.0,-0.5,0.000,0.500,0.500,0.50,-4.95,0.6,0.4,0.40,0.35,31.12,4.90,0.20,0.05,4.00,1.0
157,Jalen Williams,PTS,16.5,29.23,18.13,0.586,0.414,PTS,Betr DFS,Los Angeles Clippers,-8.0,225.0,114.9,17.0,97.33,28.0,-115.0,-105.0,0.535,0.512,15.5,16.5,7.58,-1.0,0.0,0.132,0.447,0.553,-16.43,7.97,0.4,0.5,0.60,0.72,23.71,5.00,0.26,0.08,20.25,4.0
127,Toumani Camara,PTS,14.5,31.79,11.04,0.511,0.489,PTS,Betr DFS,San Antonio Spurs,3.5,229.0,110.2,3.0,100.72,12.0,-122.0,-103.0,0.550,0.507,18.5,17.0,9.40,4.0,2.5,-0.426,0.665,0.335,21.01,-33.98,0.8,0.7,0.53,0.36,32.04,4.80,0.17,0.05,10.83,6.0
37,Jarrett Allen,REB,8.5,26.90,9.28,0.558,0.442,REB,Betr DFS,Atlanta Hawks,-3.0,236.5,112.7,9.0,102.46,6.0,-137.0,-137.0,0.578,0.578,8.8,9.5,3.46,0.8,1.5,-0.231,0.591,0.409,2.24,-29.25,0.6,0.7,0.80,0.61,25.90,5.45,0.23,0.05,9.50,4.0


In [14]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_dds, nameDict,
    line_bookmaker='DraftKings Pick6',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

draftKings_all_lines = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left').dropna()
draftKings_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q50,STAT_Q50,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
130,Donovan Clingan,PTS,12.5,25.89,10.50,0.298,0.702,PTS,DraftKings Pick6,San Antonio Spurs,3.5,229.0,110.2,3.0,100.72,12.0,-118.0,104.0,0.541,0.490,12.6,10.5,8.40,0.1,-2.0,-0.012,0.505,0.495,-6.70,0.98,0.2,0.5,0.47,0.29,26.41,3.79,0.19,0.07,9.33,6.0
113,Paolo Banchero,PTS,22.5,34.54,22.11,0.400,0.600,PTS,DraftKings Pick6,Minnesota Timberwolves,-10.5,231.5,111.9,5.0,101.46,9.0,100.0,-110.0,0.500,0.524,22.4,21.0,11.10,-0.1,-1.5,0.009,0.496,0.504,-0.80,-3.78,0.4,0.5,0.47,0.57,35.06,4.39,0.27,0.07,34.00,2.0
122,Deni Avdija,PTS,25.5,30.48,17.64,0.237,0.763,PTS,DraftKings Pick6,San Antonio Spurs,3.5,229.0,110.2,3.0,100.72,12.0,102.0,-116.0,0.495,0.537,23.6,24.0,4.62,-1.9,-1.5,0.411,0.341,0.659,-31.12,22.71,0.6,0.4,0.27,0.30,32.27,6.08,0.30,0.03,20.20,5.0
128,Scoot Henderson,PTS,14.5,27.95,15.40,0.636,0.364,PTS,DraftKings Pick6,San Antonio Spurs,3.5,229.0,110.2,3.0,100.72,12.0,-117.0,-105.0,0.539,0.512,14.6,13.5,4.77,0.1,-1.0,-0.021,0.508,0.492,-5.78,-3.94,0.6,0.4,0.53,0.38,25.76,4.22,0.24,0.04,9.00,3.0
140,Cameron Johnson,PTS,12.5,28.92,14.20,0.608,0.392,PTS,DraftKings Pick6,Memphis Grizzlies,-23.5,246.5,118.1,26.0,101.45,10.0,-108.0,-105.0,0.519,0.512,14.2,15.5,5.16,1.7,3.0,-0.329,0.629,0.371,21.14,-27.57,0.6,0.6,0.67,0.67,31.08,5.83,0.15,0.02,14.83,6.0


In [15]:
all_line_probs = pd.concat([underdog_all_lines, prizePicks_all_lines, betr_all_lines, draftKings_all_lines])
all_line_probs.to_json('data/props/ev_analysis/all_line_probs.json', orient='records', lines=True)
all_line_probs.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q50,STAT_Q50,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
12,Dylan Harper,AST,5.0,26.19,4.07,0.375,0.494,AST,PrizePicks,Portland Trail Blazers,-3.5,229.0,113.7,13.0,101.76,7.0,-137.0,-137.0,0.578,0.578,4.2,4.5,1.40,-0.8,-0.5,0.571,0.284,0.716,-50.87,23.86,0.0,0.2,0.20,0.21,24.32,2.62,0.20,0.02,2.50,2.0
47,Jalen Suggs,REB,4.5,27.95,3.79,0.431,0.569,REB,PrizePicks,Minnesota Timberwolves,-10.5,231.5,111.9,5.0,101.46,9.0,101.0,-111.0,0.498,0.526,4.5,4.0,2.22,0.0,-0.5,0.000,0.500,0.500,0.50,-4.95,0.6,0.4,0.40,0.35,31.12,4.90,0.20,0.05,4.00,1.0
79,Sam Merrill,REB,2.5,22.71,2.70,0.492,0.508,REB,Underdog,Atlanta Hawks,-3.0,236.5,112.7,9.0,102.46,6.0,-119.0,114.0,0.543,0.467,3.8,3.5,2.62,1.3,1.0,-0.496,0.690,0.310,26.98,-33.66,0.4,0.6,0.53,0.34,29.42,2.06,0.18,0.05,1.67,3.0
113,Paolo Banchero,PTS,22.5,34.54,22.11,0.400,0.600,PTS,Betr DFS,Minnesota Timberwolves,-10.5,231.5,111.9,5.0,101.46,9.0,100.0,-110.0,0.500,0.524,22.4,21.0,11.10,-0.1,-1.5,0.009,0.496,0.504,-0.80,-3.78,0.4,0.5,0.47,0.57,35.06,4.39,0.27,0.07,34.00,2.0
27,Jalen Johnson,AST,8.5,34.20,5.85,0.338,0.662,AST,DraftKings Pick6,Cleveland Cavaliers,3.0,236.5,114.1,16.0,100.63,13.0,110.0,-145.0,0.476,0.592,7.6,7.0,3.34,-0.9,-1.5,0.269,0.394,0.606,-17.26,2.39,0.2,0.4,0.47,0.30,35.90,4.14,0.26,0.03,7.75,4.0


### Get top EVs

In [16]:
slate_path = build_greedy_slate(
    prob_df=prizePicks_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/prizepicks.json",
)
print(slate_path)

Legs: 105  |  Pairs: 119  |  Slate: 5  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/prizepicks.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/prizepicks.json


In [17]:
slate_path = build_greedy_slate(
    prob_df=underdog_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/underdog.json",
)
print(slate_path)

Legs: 41  |  Pairs: 8  |  Slate: 1  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/underdog.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/underdog.json


In [18]:
slate_path = build_greedy_slate(
    prob_df=draftKings_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/draftKings.json",
)
print(slate_path)

Legs: 87  |  Pairs: 123  |  Slate: 4  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/draftKings.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/draftKings.json


In [19]:
slate_path = build_greedy_slate(
    prob_df=betr_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/betr.json",
)
print(slate_path)

Legs: 89  |  Pairs: 136  |  Slate: 6  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/betr.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/betr.json
